# Sketch-to-Image Synthesis — Pix2Pix
**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── 1. Mount Drive & set paths ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/sketch2image'
CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints/pix2pix'
SAMPLE_DIR  = f'{DRIVE_ROOT}/samples/pix2pix'

import os
os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)
print('Drive mounted.')

In [ ]:
# ── 2. Clone repo & install deps ────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/sketch2image.git /content/sketch2image
%cd /content/sketch2image
!pip install -q torchmetrics lpips

In [ ]:
# ── 3. Download dataset ─────────────────────────────────────────────────────
!python data/download_dataset.py --dataset edges2shoes --dest ./data/raw
!ls ./data/raw/edges2shoes

In [ ]:
# ── 4. Verify GPU ───────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 5. Preview dataset ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image
import os

sample_files = sorted(os.listdir('./data/raw/edges2shoes/train'))[:4]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, fname in enumerate(sample_files):
    img = Image.open(f'./data/raw/edges2shoes/train/{fname}')
    w, h = img.size
    axes[0, i].imshow(img.crop((0, 0, w//2, h)))
    axes[0, i].set_title('Sketch')
    axes[0, i].axis('off')
    axes[1, i].imshow(img.crop((w//2, 0, w, h)))
    axes[1, i].set_title('Photo')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Train ────────────────────────────────────────────────────────────────
!python train/train_pix2pix.py \
    --data_root  ./data/raw/edges2shoes \
    --save_dir   {CKPT_DIR} \
    --sample_dir {SAMPLE_DIR} \
    --img_size   256 \
    --batch_size 8 \
    --n_epochs   200 \
    --decay_epoch 100 \
    --save_every  5

In [ ]:
# ── 7. Resume from checkpoint ───────────────────────────────────────────────
# Modify train_pix2pix.py to accept --resume_ckpt if session drops.
# Quick resume snippet:
import torch
import sys; sys.path.insert(0, '/content/sketch2image')
from models.pix2pix import UNetGenerator, PatchGANDiscriminator

device = 'cuda'
ckpt = torch.load(f'{CKPT_DIR}/ckpt_epoch_0100.pt', map_location=device)

G = UNetGenerator().to(device)
D = PatchGANDiscriminator().to(device)
G.load_state_dict(ckpt['G'])
D.load_state_dict(ckpt['D'])
print(f"Resumed from epoch {ckpt['epoch']}")

In [ ]:
# ── 8. Visualise training progress ──────────────────────────────────────────
from PIL import Image
import matplotlib.pyplot as plt
import glob

samples = sorted(glob.glob(f'{SAMPLE_DIR}/*.png'))
show = samples[::len(samples)//6][:6] if len(samples) > 6 else samples

fig, axes = plt.subplots(1, len(show), figsize=(18, 4))
for ax, path in zip(axes, show):
    ax.imshow(Image.open(path))
    ax.set_title(os.path.basename(path).split('.')[0])
    ax.axis('off')
plt.suptitle('Training progression (sketch | fake | real)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9. FID evaluation ───────────────────────────────────────────────────────
!pip install -q pytorch-fid

import torch
from torchvision.utils import save_image
from data.preprocess import get_loaders

device = 'cuda'
os.makedirs('/tmp/fid_fake', exist_ok=True)
os.makedirs('/tmp/fid_real', exist_ok=True)

_, val_loader = get_loaders('./data/raw/edges2shoes', batch_size=16)
G.eval()

with torch.no_grad():
    for i, (sketch, photo) in enumerate(val_loader):
        fake = G(sketch.to(device))
        for j in range(len(fake)):
            save_image(fake[j]  * 0.5 + 0.5, f'/tmp/fid_fake/{i*16+j}.png')
            save_image(photo[j] * 0.5 + 0.5, f'/tmp/fid_real/{i*16+j}.png')

!python -m pytorch_fid /tmp/fid_real /tmp/fid_fake

In [ ]:
# ── 10. Inference on a custom sketch ────────────────────────────────────────
from google.colab import files
import torchvision.transforms as T

uploaded = files.upload()   # upload your sketch PNG
fname = list(uploaded.keys())[0]

transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

sketch_img = Image.open(fname).convert('RGB')
x = transform(sketch_img).unsqueeze(0).to(device)

G.eval()
with torch.no_grad():
    out = G(x)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sketch_img);                          axes[0].set_title('Input Sketch');      axes[0].axis('off')
axes[1].imshow((out[0].cpu().permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1))
axes[1].set_title('Generated Image'); axes[1].axis('off')
plt.tight_layout(); plt.show()